# Frequently Bought Together (v2)

This notebook reads the product pair frequency data from PostgreSQL and displays the strongest combinations in both table and chart form.

In [ ]:
import pandas as pd
import psycopg2
from psycopg2.extras import RealDictCursor

conn = psycopg2.connect(
    host='db',
    port=5432,
    dbname='zava',
    user='postgres',
    password='P@ssw0rd!'
)

sql = '''
WITH order_pairs AS (
    SELECT
        CASE
            WHEN oi1.product_id < oi2.product_id THEN oi1.product_id
            ELSE oi2.product_id
        END AS product_a,
        CASE
            WHEN oi1.product_id < oi2.product_id THEN oi2.product_id
            ELSE oi1.product_id
        END AS product_b
    FROM retail.order_items oi1
    JOIN retail.order_items oi2
      ON oi1.order_id = oi2.order_id
     AND oi1.product_id < oi2.product_id
)
SELECT
    p1.product_name AS product_a_name,
    p2.product_name AS product_b_name,
    COUNT(*) AS pair_count
FROM order_pairs op
JOIN retail.products p1
  ON p1.product_id = op.product_a
JOIN retail.products p2
  ON p2.product_id = op.product_b
GROUP BY p1.product_name, p2.product_name
HAVING COUNT(*) >= 5
ORDER BY pair_count DESC
LIMIT 20;
'''

with conn.cursor(cursor_factory=RealDictCursor) as cur:
    cur.execute(sql)
    rows = cur.fetchall()

conn.close()

summary = pd.DataFrame(rows)
summary

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

top = summary.head(10)
plt.figure(figsize=(12, 7))
bars = plt.bar(top['product_a_name'] + ' + ' + top['product_b_name'], top['pair_count'], color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'])
plt.title('Top Product Pair Counts')
plt.xlabel('Product Pair')
plt.ylabel('Pair Count')
plt.xticks(rotation=35, ha='right')
plt.grid(axis='y', alpha=0.3)

for bar, value in zip(bars, top['pair_count']):
    plt.text(bar.get_x() + bar.get_width() / 2, value + 0.5, f'{value}', ha='center', va='bottom')

plt.tight_layout()
plt.show()